# Exp03-Part2-t2.基于ResNet18网络完成图像分类任务（Pytorch）

在本实践中，我们实践一个更通用的图像分类任务。

**图像分类**（Image Classification）是计算机视觉中的一个基础任务，将图像的语义将不同图像划分到不同类别。很多任务也可以转换为图像分类任务。比如人脸检测就是判断一个区域内是否有人脸，可以看作一个二分类的图像分类任务。

这里，我们使用的计算机视觉领域的经典数据集：CIFAR-10数据集，网络为ResNet18模型，损失函数为交叉熵损失，优化器为Adam优化器，评价指标为准确率。

*******

Adam优化器的介绍参考《神经网络与深度学习》第7.2.4.3节。

*******

In [ ]:
import datetime
import os

def printLog(name, ID):
    # 打印和
    print(': %s; : %s' % (name, ID))
    # 打印操作系统的用户名
    print("系统用户名:", os.getlogin())
    # 打印当前时间
    print('运行时间:', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

"""*********************************************************************
   【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。
   *********************************************************************"""
printLog('', )

# 1. 数据处理

## 1.1 数据集介绍

CIFAR-10数据集包含了10种不同的类别、共60,000张图像，其中每个类别的图像都是6000张，图像大小均为$32\times32$像素。CIFAR-10数据集的示例如 **图1** 所示。

<center><img src="https://ai-studio-static-online.cdn.bcebos.com/12cb79a15a3d40fa9b0124dfe4681ab8b60b82d067fb433eb6da600b7c2ec875" width="600" hegiht="" ></center>
<center><br>图1：CIFAR-10数据集示例</br></center>
<br></br>


## 1.2 自定义数据集
### 1.2.1 数据读取

在本实践中，将原始训练集拆分成了train_set、dev_set两个部分，分别包括40 000条和10 000条样本。将data_batch_1到data_batch_4作为训练集，data_batch_5作为验证集，test_batch作为测试集。
最终的数据集构成为：

* 训练集：40 000条样本。
* 验证集：10 000条样本。
* 测试集：10 000条样本。

读取一个batch数据的代码如下所示：

In [ ]:
import os
import pickle
import numpy as np

def load_cifar10_batch(folder_path, batch_id=1, mode='train'):
    if mode == 'test':
        file_path = os.path.join(folder_path, 'test_batch')
    else:
        file_path = os.path.join(folder_path, 'data_batch_'+str(batch_id))

    # 加载数据集文件
    with open(file_path, 'rb') as batch_file:
        batch = pickle.load(batch_file, encoding='latin1')

    imgs = batch['data'].reshape((len(batch['data']), 3, 32, 32)) / 255.
    labels = batch['labels']

    return np.array(imgs, dtype='float32'), np.array(labels)

# 使用您的实际文件路径
folder_path = r'~\anaconda3\数据文件\cifar-10-batches-py'

# 检查路径是否存在
if not os.path.exists(folder_path):
    print(f"错误：路径 {folder_path} 不存在")
    # 列出可能的目录
    parent_dir = r'~\anaconda3\数据文件'
    if os.path.exists(parent_dir):
        print("该目录下的文件：")
        for item in os.listdir(parent_dir):
            print(f"  {item}")
else:
    try:
        imgs_batch, labels_batch = load_cifar10_batch(
            folder_path=folder_path, 
            batch_id=1, 
            mode='train'
        )
        print(f"成功加载批次！图像形状：{imgs_batch.shape}, 标签形状：{labels_batch.shape}")
        print(f"像素值范围：[{imgs_batch.min():.3f}, {imgs_batch.max():.3f}]")
        print(f"前10个标签：{labels_batch[:10]}")
    except Exception as e:
        print(f"加载数据时出错：{e}")
        # 检查目录内容
        print("\n目录内容：")
        for item in os.listdir(folder_path):
            print(f"  {item}")

查看数据的维度：

In [ ]:
#打印一下每个batch中X和y的维度
print ("batch of imgs shape: ",imgs_batch.shape, "batch of labels shape: ", labels_batch.shape)

可视化观察其中的一张样本图像和对应的标签，代码如下所示：

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

image, label = imgs_batch[1], labels_batch[1]
print("The label in the picture is {}".format(label))
plt.figure(figsize=(2, 2))
# print(image.shape)
# print(image.transpose(1,2,0).shape)
plt.imshow(image.transpose(1,2,0)) # numpy数组，https://blog.csdn.net/trayvontang/article/details/108391139
plt.savefig('cnn-car.pdf')

### 1.2.2 构造Dataset类

构造一个CIFAR10Dataset类，其将继承自`paddle.io.DataSet`类，可以逐个数据进行处理。代码实现如下：

In [ ]:
import os
import pickle
import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision.transforms import Compose, ToTensor, Normalize

def load_cifar10_batch(folder_path, batch_id=1, mode='train'):
    if mode == 'test':
        file_path = os.path.join(folder_path, 'test_batch')
    else:
        file_path = os.path.join(folder_path, 'data_batch_'+str(batch_id))

    # 加载数据集文件
    with open(file_path, 'rb') as batch_file:
        batch = pickle.load(batch_file, encoding='latin1')

    imgs = batch['data'].reshape((len(batch['data']), 3, 32, 32)) / 255.
    labels = batch['labels']

    return np.array(imgs, dtype='float32'), np.array(labels)

class CIFAR10Dataset(Dataset):
    def __init__(self, folder_path='/datasets/cifar-10-batches-py', mode='train', transform=None):
        self.mode = mode
        self.folder_path = folder_path
        
        if mode == 'train':
            # 加载batch1-batch4作为训练集
            self.imgs, self.labels = load_cifar10_batch(folder_path=self.folder_path, batch_id=1, mode='train')
            for i in range(2, 5):
                imgs_batch, labels_batch = load_cifar10_batch(folder_path=self.folder_path, batch_id=i, mode='train')
                self.imgs = np.concatenate([self.imgs, imgs_batch])
                self.labels = np.concatenate([self.labels, labels_batch])
        elif mode == 'dev':
            # 加载batch5作为验证集
            self.imgs, self.labels = load_cifar10_batch(folder_path=self.folder_path, batch_id=5, mode='train')
        elif mode == 'test':
            # 加载测试集
            self.imgs, self.labels = load_cifar10_batch(folder_path=self.folder_path, mode='test')
        
        # 如果没有提供transform，使用默认的transform
        if transform is None:
            self.transform = Compose([
                ToTensor(),
                Normalize(mean=[0.4914, 0.4822, 0.4465], 
                         std=[0.2023, 0.1994, 0.2010])
            ])
        else:
            self.transform = transform

    def __getitem__(self, idx):
        img, label = self.imgs[idx], self.labels[idx]
        
        # 将numpy数组转换为PIL图像格式（CHW->HWC）以便ToTensor处理
        img = img.transpose(1, 2, 0)  # 从CHW转为HWC
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

    def __len__(self):
        return len(self.imgs)


# 设置数据路径 - 使用您的实际路径
folder_path = r'~\anaconda3\数据文件\cifar-10-batches-py'

# 检查路径是否存在
if not os.path.exists(folder_path):
    print(f"错误：路径 {folder_path} 不存在")
else:
    # 设置随机种子
    torch.manual_seed(100)
    
    # 创建数据集
    train_dataset = CIFAR10Dataset(folder_path=folder_path, mode='train')
    dev_dataset = CIFAR10Dataset(folder_path=folder_path, mode='dev')
    test_dataset = CIFAR10Dataset(folder_path=folder_path, mode='test')
    
    print("数据集创建成功！")
    print(f"训练集大小: {len(train_dataset)}")
    print(f"验证集大小: {len(dev_dataset)}")
    print(f"测试集大小: {len(test_dataset)}")
    
    # 测试一个样本
    sample_img, sample_label = train_dataset[0]
    print(f"样本图像形状: {sample_img.shape}")
    print(f"样本标签: {sample_label}")
    print(f"样本标签类型: {type(sample_label)}")

"""*********************************************************************
   【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。
   *********************************************************************"""
printLog('', )

## 1.3 pytorch内置数据集（本实践中不使用）

__此处仅做简要介绍，本实践使用1.2节中的自定义数据集，实际应用中都需要自定义数据集。__

In [ ]:
import torchvision

print('PyTorch内置数据集：', torchvision.datasets.__all__)

**准备训练集和测试集**

准备训练集cifar10_train和测试集cifar10_test

`torchvision.datasets.CIFAR()`中的`train=True`表示加载训练集，`train=False`表示加载测试集，分别用于获取Cifar10训练集和测试集

规范化： 通过规范化手段，把输入图像的分布改变成均值为0，标准差为1的标准正态分布，使得最优解的寻优过程明显会变得平缓，训练过程更容易收敛。
*******

在飞桨中，提供了部分视觉领域的高层API，可以直接调用API实现简单的图像处理操作。通过调用`torchvision.transforms.Resize`调整大小；调用`torchvision.transforms.Normalize`进行标准化处理；使用`torchvision.transforms.Compose`将两个预处理操作进行拼接。

*******

In [ ]:
import torch
from torchvision import datasets, transforms

# 定义数据预处理转换操作
# 使用Compose将多个转换操作串联起来
transform = transforms.Compose([
    # 将PIL图像或NumPy数组转换成PyTorch张量，并自动归一化到[0,1]
    transforms.ToTensor(),
    # 对图像进行标准化处理，使用CIFAR10数据集的均值和标准差
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], 
                         std=[0.2023, 0.1994, 0.2010])
])


cifar10_train = datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

# 加载 CIFAR-10 数据集的测试部分
# train=False: 表示加载测试集
cifar10_test = datasets.CIFAR10(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

# 打印数据集信息
print("训练集大小:", len(cifar10_train))
print("测试集大小:", len(cifar10_test))
print("\n类别标签:", cifar10_train.classes)
print("类别到索引的映射:", cifar10_train.class_to_idx)

# 查看一个样本
sample_img, sample_label = cifar10_train[0]
print(f"\n样本图像形状: {sample_img.shape}")
print(f"样本标签索引: {sample_label}")
print(f"对应的标签名称: {cifar10_train.classes[sample_label]}")
print(f"像素值范围: [{sample_img.min():.3f}, {sample_img.max():.3f}]")
print(f"像素均值: {sample_img.mean():.3f}, 标准差: {sample_img.std():.3f}")

# 2. 模型构建

使用torchvision中的Resnet18进行图像分类实践。

In [ ]:
import torchvision.models as models

# 创建一个ResNet-18模型，输出类别数为10
resnet18_model = models.resnet18(num_classes=10)


# 3. 模型训练

接下来，用一个循环来进行模型的训练，将会:
* 使用 `torch.utils.data.DataLoader` 来加载数据并组建batch。
* 使用 `nn.CrossEntropyLoss()` 来计算损失值。
* 使用 `torch.optim` 优化器来进行优化。

在实践中，保存准确率最高的模型作为最佳模型。

用DataLoader对数据集进行封装

In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision.models as models
# 批量大小
batch_size = 64

# 加载数据
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
import torch.optim as optim
import torch.nn as nn

# 指定运行设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

resnet18_model = resnet18_model.to(device)
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet18_model.parameters(), lr=learning_rate)

# 初始化记录列表
train_losses = []        # 记录每训练1个epoch后，模型在整个训练集上的平均损失
train_step_losses = []   # 记录每训练1个step后，模型在该batch训练数据上的平均损失
val_losses = []          # 记录模型在验证集上的平均损失        
val_accuracies = []      # 记录模型在验证集上的精度

# 训练模型：使用训练集和验证集进行模型训练，共训练 20个epoch。保存准确率最高的模型作为最佳模型。
num_epochs = 20           # 训练6个epoch准确率还比较低
best_accuracy = 0.0       # 最佳准确率
best_model_state = None   # 最佳模型的状态字典
# 模型保存路径
model_path = "best_model.pth"

# 打印GPU信息（如果可用）
if torch.cuda.is_available():
    print(f"GPU名称: {torch.cuda.get_device_name(0)}")
    print(f"GPU内存总量: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

使用训练集和验证集进行模型训练，共训练20个epoch。在实践中，保存准确率最高的模型作为最佳模型。代码实现如下：

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 检查是否有可用的GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
if torch.cuda.is_available():
    print(f"GPU名称: {torch.cuda.get_device_name(0)}")

# 定义数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], 
                        std=[0.2023, 0.1994, 0.2010])
])

# 批量大小
batch_size = 64

# 使用torchvision的CIFAR10数据集
from torchvision.datasets import CIFAR10

train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)

# 划分训练集和验证集（80%训练，20%验证）
from torch.utils.data import random_split

# 设置随机种子以确保可重复性
torch.manual_seed(42)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_dataset = random_split(train_dataset, [train_size, val_size])

# 创建数据加载器
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("数据集信息:")
print(f"训练集大小: {len(train_subset)}")
print(f"验证集大小: {len(val_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

# 创建一个ResNet-18模型，输出类别数为10
resnet18_model = models.resnet18(num_classes=10)
resnet18_model = resnet18_model.to(device)

# 设置学习率
learning_rate = 0.001

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet18_model.parameters(), lr=learning_rate)

# 初始化记录列表
train_losses = []        # 记录每训练1个epoch后，模型在整个训练集上的平均损失
val_losses = []          # 记录模型在验证集上的平均损失        
val_accuracies = []      # 记录模型在验证集上的精度

# 训练参数
num_epochs = 20
best_accuracy = 0.0
best_model_state = None
model_path = "best_model.pth"

# 训练循环
print("\n开始训练...")
for epoch in range(num_epochs):
    # 训练阶段
    resnet18_model.train()
    train_epoch_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # 前向传播
        optimizer.zero_grad()
        outputs = resnet18_model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播和优化
        loss.backward()
        optimizer.step()
        
        # 统计训练损失和准确率
        train_epoch_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()
        
        # 每100个batch打印一次训练信息
        if (batch_idx + 1) % 100 == 0:
            batch_acc = 100.0 * correct_train / total_train
            print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], '
                  f'Loss: {loss.item():.4f}, Acc: {batch_acc:.2f}%')
    
    # 计算训练集平均损失和准确率
    train_epoch_loss /= len(train_loader)
    train_accuracy = 100.0 * correct_train / total_train
    train_losses.append(train_epoch_loss)
    
    # 验证阶段
    resnet18_model.eval()
    val_epoch_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = resnet18_model(images)
            loss = criterion(outputs, labels)
            
            val_epoch_loss += loss.item()
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()
    
    # 计算验证集平均损失和准确率
    val_epoch_loss /= len(val_loader)
    val_accuracy = 100.0 * correct_val / total_val
    val_losses.append(val_epoch_loss)
    val_accuracies.append(val_accuracy)
    
    # 打印每个epoch的结果
    print(f'Epoch [{epoch+1}/{num_epochs}] 完成: '
          f'Train Loss: {train_epoch_loss:.4f}, Train Acc: {train_accuracy:.2f}%, '
          f'Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
    
    # 保存最佳模型
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model_state = resnet18_model.state_dict()
        torch.save(best_model_state, model_path)
        print(f'  最佳模型已保存，准确率: {best_accuracy:.2f}%')

# 训练完成
print(f"\n训练完成！最佳验证准确率: {best_accuracy:.2f}%")

# 加载最佳模型进行测试
if best_model_state is not None:
    resnet18_model.load_state_dict(best_model_state)

# 在测试集上评估模型
print("\n在测试集上评估模型...")
resnet18_model.eval()
test_loss = 0.0
correct_test = 0
total_test = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = resnet18_model(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()

# 计算测试准确率
test_loss /= len(test_loader)
test_accuracy = 100.0 * correct_test / total_test

print(f"测试结果:")
print(f"测试损失: {test_loss:.4f}")
print(f"测试准确率: {test_accuracy:.2f}%")

# 显示类别信息
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']
print(f"\nCIFAR-10类别: {class_names}")

"""*********************************************************************
   【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。
   *********************************************************************"""
printLog('', )  

可视化观察训练集与验证集的准确率及损失变化情况。

In [ ]:
import matplotlib.pyplot as plt

# 绘制损失变化曲线
plt.figure(figsize=(12, 5))

# 子图1：训练损失
plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='Train Loss')
plt.plot(range(1, len(val_losses) + 1), val_losses, 'r-', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# 子图2：验证准确率
plt.subplot(1, 2, 2)
plt.plot(range(1, len(val_accuracies) + 1), val_accuracies, 'g-', label='Val Accuracy')
plt.axhline(y=best_accuracy, color='r', linestyle='--', label=f'Best: {best_accuracy:.2f}%')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 打印最佳结果
print(f"\n最佳验证准确率: {best_accuracy:.2f}%")
print(f"最终测试准确率: {test_accuracy:.2f}%")

# 也可以绘制损失和准确率的组合图
plt.figure(figsize=(10, 8))

# 组合图：双Y轴
fig, ax1 = plt.subplots(figsize=(10, 6))

color = 'tab:blue'
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color=color)
ax1.plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='Train Loss', alpha=0.7)
ax1.plot(range(1, len(val_losses) + 1), val_losses, 'b--', label='Val Loss', alpha=0.7)
ax1.tick_params(axis='y', labelcolor=color)
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# 创建第二个Y轴用于准确率
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Accuracy (%)', color=color)
ax2.plot(range(1, len(val_accuracies) + 1), val_accuracies, 'g-', label='Val Accuracy')
ax2.axhline(y=best_accuracy, color='r', linestyle='--', label=f'Best: {best_accuracy:.2f}%')
ax2.tick_params(axis='y', labelcolor=color)
ax2.legend(loc='upper right')

plt.title('Training Loss and Validation Accuracy')
fig.tight_layout()
plt.show()

# 打印详细的性能指标
print("\n" + "="*50)
print("训练性能摘要:")
print("="*50)
print(f"训练轮次: {num_epochs}")
print(f"最佳验证准确率: {best_accuracy:.2f}% (第{val_accuracies.index(best_accuracy)+1}轮)")
print(f"最终测试准确率: {test_accuracy:.2f}%")
print(f"训练损失变化: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}")
print(f"验证损失变化: {val_losses[0]:.4f} -> {val_losses[-1]:.4f}")
print(f"验证准确率变化: {val_accuracies[0]:.2f}% -> {val_accuracies[-1]:.2f}%")
print("="*50)

"""*********************************************************************
   【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。
   *********************************************************************"""
printLog('', )  

*******

在本实践中，使用了第7章中介绍的Adam优化器进行网络优化，如果使用SGD优化器，会造成过拟合的现象，在验证集上无法得到很好的收敛效果。可以尝试使用第7章中其他优化策略调整训练配置，达到更高的模型精度。

*******

# 4. 模型评价

使用测试数据对在训练过程中保存的最佳模型进行评价，观察模型在测试集上的准确率以及损失情况。代码实现如下：

In [ ]:
import os
import torch

# 加载最佳模型
if os.path.exists(model_path):
    resnet18_model.load_state_dict(torch.load(model_path))
    print(f"已加载最佳模型: {model_path}")
else:
    print("警告: 找不到最佳模型文件，使用当前模型进行测试")

# 测试模型
resnet18_model.eval()  # 设置模型为评估模式
test_loss = 0.0
correct = 0
total = 0

# 使用测试数据加载器
with torch.no_grad():  # 不计算梯度，节省内存和计算资源
    for batch_idx, (images, labels) in enumerate(test_loader):
        # 将数据移动到设备上
        images, labels = images.to(device), labels.to(device)
        
        # 前向传播
        outputs = resnet18_model(images)
        
        # 计算损失
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        
        # 计算准确率
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # 每50个batch打印一次进度
        if (batch_idx + 1) % 50 == 0:
            batch_acc = 100.0 * (predicted == labels).sum().item() / labels.size(0)
            print(f'测试进度: [{batch_idx+1}/{len(test_loader)}], '
                  f'当前batch准确率: {batch_acc:.2f}%')

# 计算平均测试损失和准确率
average_test_loss = test_loss / len(test_loader)
test_accuracy = 100.0 * correct / total

print("\n" + "="*50)
print("测试结果汇总:")
print("="*50)
print(f"测试集大小: {total}")
print(f"平均测试损失: {average_test_loss:.4f}")
print(f"测试准确率: {test_accuracy:.2f}%")
print(f"正确预测的样本数: {correct}/{total}")
print("="*50)

# 如果需要，还可以计算每个类别的准确率
print("\n按类别分析:")
print("-"*50)

# 获取类别名称
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# 初始化每个类别的统计
class_correct = [0] * 10
class_total = [0] * 10

# 重新运行一次测试来统计每个类别的准确率
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet18_model(images)
        _, predicted = torch.max(outputs, 1)
        
        c = (predicted == labels).squeeze()
        for i in range(len(labels)):
            label = labels[i].item()
            class_correct[label] += c[i].item()
            class_total[label] += 1

# 打印每个类别的准确率
for i in range(10):
    if class_total[i] > 0:
        class_acc = 100.0 * class_correct[i] / class_total[i]
        print(f'{class_names[i]:<12s}: {class_acc:.2f}% ({class_correct[i]}/{class_total[i]})')
    else:
        print(f'{class_names[i]:<12s}: 0.00% (0/0)')

# 可视化部分样本的预测结果
print("\n可视化部分测试样本的预测结果:")
print("-"*50)

# 获取一批测试数据
test_batch = next(iter(test_loader))
images, labels = test_batch
images, labels = images.to(device), labels.to(device)

# 进行预测
with torch.no_grad():
    outputs = resnet18_model(images[:8])  # 只取前8个样本
    _, predicted = torch.max(outputs, 1)

# 将图像移回CPU并转换为numpy
images_np = images[:8].cpu()
predicted_labels = predicted.cpu().numpy()
true_labels = labels[:8].cpu().numpy()

# 打印前8个样本的预测结果
for i in range(8):
    pred_name = class_names[predicted_labels[i]]
    true_name = class_names[true_labels[i]]
    correct_str = "✓" if predicted_labels[i] == true_labels[i] else "✗"
    print(f'样本 {i+1}: 预测={pred_name:<12s}, 真实={true_name:<12s} {correct_str}')

"""*********************************************************************
   【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。
   *********************************************************************"""
printLog('', )  

# 5. 模型预测

同样地，也可以使用保存好的模型，对测试集中的数据进行模型预测，观察模型效果，具体代码实现如下：

In [ ]:
import torch.nn.functional as F

# 获取测试集中的一个 batch 的数据
X, label = next(iter(test_loader))

# 确保输入数据和标签都在正确的设备上
device = next(model.parameters()).device  # 获取模型所在设备
X = X.to(device)
label = label.to(device)

# 假设 model 已经定义并且已经加载了权重
model.eval()  # 设置模型为评估模式
with torch.no_grad():  # 不需要梯度计算
    logits = model(X)
    
# 多分类，使用 softmax 计算预测概率
pred = F.softmax(logits, dim=1)
# 获取概率最大的类别
pred_class = torch.argmax(pred[2]).item()
# 获取标签
label = label[2].item()

# 输出真实类别与预测类别
print(f"The true category is {label} and the predicted category is {pred_class}")

#可视化图片
plt.figure(figsize=(2, 2))
imgs, labels = load_cifar10_batch(folder_path='datasets/cifar-10-batches-py', mode='test')
plt.imshow(imgs[2].transpose(1,2,0))
plt.savefig('cnn-test-vis.pdf')

# 6. 实践拓展

* 尝试加深残差网络的层数或使用《神经网络与深度学习》中介绍的其他模型完成基于CIFAR10的图像分类实践，观察是否能够得到更高的精度；
* 尝试使用CIFAR-100进行实践，观察不同模型在不同数据集上的学习效果；